# SME Liquidity GRPO Training on Colab

This is the judge-facing thin wrapper around the repository's canonical environment-native trainer. It trains against the in-process SME liquidity OpenEnv workflow, saves dashboard artifacts, and evaluates base vs trained policies on fixed seeds.

## 1. Install Dependencies

The pinned TRL version is the tested Colab path for this submission.

In [ ]:
%pip install -q "trl==0.29.0" "vllm>=0.11.0" "peft>=0.17.0" "transformers>=4.56.0,<4.57.0" "huggingface_hub>=0.34.0,<1.0" "accelerate>=1.0.0" bitsandbytes sentencepiece matplotlib datasets

from pathlib import Path

REPO_URL = "https://github.com/SkandaGanesha1/OpenEnv_SME_Negotiator-2.o.git"
REPO_DIR = Path("OpenEnv_SME_Negotiator-2.o")

if not REPO_DIR.exists():
    !git clone -q $REPO_URL

%cd $REPO_DIR
%pip install -q -e .[rl]


## 2. Configure Submission Profile

The default model is small enough for Colab while still exercising the same environment-native trainer path used by the repo.

In [ ]:
from pathlib import Path

from rl.demo import evaluate_before_after_policies, save_training_dashboard
from rl.train_grpo_liquidity import build_arg_parser, build_training_session, run_training

MODEL_NAME = "Qwen/Qwen3-0.6B"
RUN_PROFILE = "submission"
TASK_NAME = "liquidity-correlation-hard"
OUTPUT_DIR = Path("outputs/grpo_sme_liquidity_colab")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

parser = build_arg_parser()
args = parser.parse_args([
    "--profile", RUN_PROFILE,
    "--model-name", MODEL_NAME,
    "--task-name", TASK_NAME,
    "--difficulty", "hard",
    "--total-periods", "3",
    "--output-dir", str(OUTPUT_DIR),
    "--eval-num-seeds", "3",
])

session = build_training_session(args)
print("Built environment-native trainer session")
print(session.keys())


## 3. Train

This cell runs the shared training entrypoint. Saved artifacts include `reward_curve.png`, `training_dashboard.png`, `policy_comparison.png`, and `eval_summary.json`.

In [ ]:
result = run_training(args)

dashboard = save_training_dashboard(result["trainer"], output_dir=str(OUTPUT_DIR))
print("training_dashboard.png", dashboard.get("training_dashboard_path"))
print("reward_curve.png", dashboard.get("reward_curve_path"))
print("policy_comparison.png", result.get("policy_comparison_path"))
print("eval_summary.json", result.get("eval_summary_path"))

result


## 4. Evaluate Base vs Trained Policy

The final judge check compares base and trained policies on the same seeds and verifies that the trained policy improves without introducing extra defaults.

In [ ]:
comparison = evaluate_before_after_policies(
    output_dir=str(OUTPUT_DIR),
    seeds=[4100, 4101, 4102],
    total_periods=3,
    task_name=TASK_NAME,
    difficulty="hard",
    checkpoint_path=str(result["checkpoint_path"]),
    model_name=MODEL_NAME,
    policy="base",
)

trained_comparison = evaluate_before_after_policies(
    output_dir=str(OUTPUT_DIR),
    seeds=[4100, 4101, 4102],
    total_periods=3,
    task_name=TASK_NAME,
    difficulty="hard",
    checkpoint_path=str(result["checkpoint_path"]),
    model_name=MODEL_NAME,
    policy="trained",
)

summary = comparison.get("summary", {}) if isinstance(comparison, dict) else {}
_baseline_success = bool(summary.get("success_no_default_positive_npv", False))
submission_ready = bool(
    comparison.get("submission_ready", False)
    if isinstance(comparison, dict)
    else False
)

print("baseline_success", _baseline_success)
print("submission_ready", submission_ready)
comparison, trained_comparison
